# Five-league coverage and frozen-model evaluation

Real retained, checksum-pinned FBref/worldfootballR archives expand La Liga to Premier League, Bundesliga, Serie A and Ligue 1. Existing model artifacts and training transforms are unchanged. No synthetic observations, estimates, new Transfermarkt collection or transfer-success labels are introduced.

In [1]:
from pathlib import Path
import os,sys,json
ROOT=Path.cwd()
while not (ROOT/'src/config.py').exists():
    if ROOT==ROOT.parent: raise RuntimeError('Run inside the repository')
    ROOT=ROOT.parent
os.chdir(ROOT);sys.path.insert(0,str(ROOT))
import pandas as pd
from IPython.display import display,Markdown
from src.data.provenance import sha256
report=json.loads((ROOT/'reports/top_five_leagues/manifest.json').read_text())
for item in report['outputs'].values(): assert sha256(ROOT/item['path'])==item['sha256']
for filename,digest in report['report_sha256'].items(): assert sha256(ROOT/'reports/top_five_leagues'/filename)==digest
coverage=pd.read_csv(ROOT/'reports/top_five_leagues/coverage.csv')
availability=pd.read_csv(ROOT/'reports/top_five_leagues/availability.csv')
display(availability.pivot(index=['model','season'],columns='league',values='players'))


league               Bundesliga  La Liga  Ligue 1  Premier League  Serie A
model     season                                                          
core      2020-2021       338.0    393.0    377.0           369.0    383.0
          2021-2022       322.0    396.0    382.0           368.0    395.0
          2022-2023       328.0    385.0    377.0           377.0    394.0
          2023-2024       324.0    391.0    334.0           371.0    390.0
          2024-2025         NaN      1.0      1.0             NaN      2.0
offensive 2020-2021       338.0    393.0    377.0           369.0    383.0
          2021-2022       322.0    396.0    382.0           368.0    395.0
          2022-2023       328.0    385.0    377.0           377.0    394.0
          2023-2024       324.0    391.0    334.0           371.0    390.0
          2024-2025       320.0    397.0    334.0           361.0    392.0

## Category and exposure reliability

Source player ID + league + season + club defines a stint. Auxiliary category totals are used only when their reported 90s agree with standard minutes/90 within 0.061, accommodating the source rounding. Incompatible values become unknown, never zero. A multi-club season sums only within one league and missing stints propagate. Cross-league transfers retain separate historical rows; the gallery takes the highest-minutes league before feature eligibility, with alphabetical league ties. It does not blend league totals or treat the player as two distinct candidates. This rule can exclude otherwise eligible secondary-league observations and slightly changes the La Liga cohort.

In [2]:
display(coverage.groupby(['league','category']).agg(raw_stints=('raw_stints','sum'),matched=('matched','sum'),compatible=('compatible','sum')))
quality=availability[~availability.offered]
display(quality)
display(Markdown(f"Cross-league player-season rows requiring gallery selection: **{report['cross_league_person_seasons']}**. Core galleries cover 2020/21–2023/24; offensive galleries extend through 2024/25. Tiny Core 2024/25 cohorts are not offered. The 2025/26 archive is partial and excluded from recommendations. Category presence alone does not establish aligned feature completeness."))


raw_stints  matched  compatible
league         category                                   
Bundesliga     defense           2429     2429        2064
               misc              2429     2429        2064
               passing           2898     2898        2898
               possession        2429     2429        2064
               shooting          2898     2898        2898
               standard          2898     2898        2898
La Liga        defense           2874     2874        2434
               misc              2874     2874        2434
               passing           3443     3443        3434
               possession        2874     2874        2434
               shooting          3443     3443        3443
               standard          3443     3443        3443
Ligue 1        defense           2750     2750        2372
               misc              2750     2750        2372
               passing           3276     3276        3275
               possession        2750     2750        2372
               shooting          3276     3276        3275
               standard          3276     3276        3276
Premier League defense           2666     2666        2254
               misc              2666     2666        2254
               passing           3205     3205        3205
               possession        2666     2666        2254
               shooting          3205     3205        3205
               standard          3205     3205        3205
Serie A        defense           2945     2945        2508
               misc              2945     2945        2508
               passing           3512     3512        3512
               possession        2945     2945        2508
               shooting          3512     3512        3512
               standard          3512     3512        3512

,model,league,season,players,offered,advanced_variables_missing
8,core,La Liga,2024-2025,1,False,0
13,core,Ligue 1,2024-2025,1,False,0
22,core,Serie A,2024-2025,2,False,0


Cross-league player-season rows requiring gallery selection: **690**. Core galleries cover 2020/21–2023/24; offensive galleries extend through 2024/25. Tiny Core 2024/25 cohorts are not offered. The 2025/26 archive is partial and excluded from recommendations. Category presence alone does not establish aligned feature completeness.

## External league profile retrieval

The original model and neighbor method were selected on La Liga. Apply their frozen transforms and the selected method to each league without refitting on future seasons. Core queries 2022/23 against 2023/24; offensive queries 2023/24 against 2024/25. Candidate pools are league-specific and the target is the same player in the next season. All available profile methods and goals/assists/random baselines are evaluated on identical returning players. Confidence intervals are player bootstrap intervals conditional on this source cohort. Differences between Core and offensive are not controlled because features, years and cohorts differ.

In [3]:
evaluation=pd.read_csv(ROOT/'reports/top_five_leagues/evaluation.csv')
selected=evaluation[evaluation.selected_method]
display(selected[['model','league','queries','candidate_count','mrr','mrr_ci_low','mrr_ci_high','recall_at_5','mrr_delta_goals_assists','mrr_delta_ci_low','mrr_delta_ci_high']])
display(evaluation[evaluation.method.isin(['goals_assists','random'])][['model','league','method','mrr','recall_at_5']])
assert selected.query_season.lt(selected.evaluation_season).all()
display(Markdown('Observed retrieval scores describe profile persistence. The league-specific pools do not validate a pooled cross-league tactical ranking, adjust league strength, establish a signing benefit or calibrate affinity as probability. No new training-performance gain is claimed.'))


,model,league,queries,candidate_count,mrr,mrr_ci_low,mrr_ci_high,recall_at_5,mrr_delta_goals_assists,mrr_delta_ci_low,mrr_delta_ci_high
2,core,La Liga,245,391,0.222887,0.180152,0.256657,0.338776,0.183556,0.138236,0.217867
12,core,Premier League,232,371,0.179393,0.148605,0.217021,0.250000,0.133680,0.101100,0.175791
22,core,Bundesliga,210,324,0.200240,0.164988,0.240437,0.304762,0.170578,0.136248,0.212429
32,core,Serie A,240,390,0.216076,0.181682,0.261394,0.283333,0.184974,0.148070,0.228474
42,core,Ligue 1,184,334,0.235245,0.188311,0.280111,0.260870,0.187298,0.142776,0.229923
52,offensive,La Liga,243,397,0.085564,0.061735,0.111595,0.106996,0.029881,0.004710,0.058496
62,offensive,Premier League,234,361,0.119076,0.089152,0.144192,0.179487,0.067878,0.042309,0.094950
72,offensive,Bundesliga,201,320,0.086732,0.061148,0.113849,0.104478,0.041397,0.022894,0.063733
82,offensive,Serie A,243,392,0.089370,0.068085,0.118626,0.106996,0.046551,0.025673,0.072088
92,offensive,Ligue 1,183,334,0.067114,0.053024,0.085151,0.092896,0.021400,0.000705,0.042813


,model,league,method,mrr,recall_at_5
3,core,La Liga,goals_assists,0.039331,0.047279
9,core,La Liga,random,0.014733,0.008163
13,core,Premier League,goals_assists,0.045714,0.055603
19,core,Premier League,random,0.017343,0.008621
23,core,Bundesliga,goals_assists,0.029663,0.022924
29,core,Bundesliga,random,0.016004,0.014286
33,core,Serie A,goals_assists,0.031103,0.032143
39,core,Serie A,random,0.009550,0.004167
43,core,Ligue 1,goals_assists,0.047946,0.056818
49,core,Ligue 1,random,0.014205,0.010870


Observed retrieval scores describe profile persistence. The league-specific pools do not validate a pooled cross-league tactical ranking, adjust league strength, establish a signing benefit or calibrate affinity as probability. No new training-performance gain is claimed.

## App scope and valuation coverage

The five-league market is the default; the original La Liga-only gallery remains selectable. League selection restricts rankings, similar players, clusters and comparison choices; scenarios have an explicit league requirement. All six measured Core styles remain available. Offensive supports vertical and counterattack. Retained supported valuation links remain provisional: an unmatched player has unknown value, not an estimate or free-transfer status. Private uploads are not necessary to launch ordinary five-league scouting from Git.

In [4]:
from src.app.leagues import load_top_five_service
for name in ['core','offensive']:
    service=load_top_five_service(name)
    gallery=service.for_season(service.available_seasons[0]).players
    assert gallery.entity_id.is_unique
    display(gallery.groupby('league').agg(players=('entity_id','size'),median_age=('age','median'),median_minutes=('minutes','median')))
    display(Markdown(f"{name}: styles **{', '.join(service.supported_styles)}**. Dataset `{report['dataset_version']}`."))


,players,median_age,median_minutes
league,,,
Bundesliga,324,26.0,1580.0
La Liga,391,27.0,1634.0
Ligue 1,334,25.0,1557.0
Premier League,371,26.0,1692.0
Serie A,390,26.0,1651.0


core: styles **posesion, presion_alta, vertical, contraataque, solidez_defensiva, desborde_banda**. Dataset `52337690992d5e186181ccbaa12554c52b1492adfbfd79effe184ceed818821f`.

,players,median_age,median_minutes
league,,,
Bundesliga,320,26.0,1599.0
La Liga,397,26.0,1590.0
Ligue 1,334,24.5,1519.5
Premier League,361,25.0,1774.0
Serie A,392,25.0,1539.0


offensive: styles **vertical, contraataque**. Dataset `52337690992d5e186181ccbaa12554c52b1492adfbfd79effe184ceed818821f`.